**Machine learning** is a subfield of artificial intelligence where systems learn patterns from data and make decisions — without being explicitly programmed for every scenario.

Instead of writing rules by hand, you give a machine learning model examples, and it figures out the rules itself.

In this notebook we'll cover:
- What machine learning is and why it matters
- How it differs from traditional programming
- The three main types of ML
- When to use (and not use) ML
- Key terminology you'll encounter throughout this series

## Traditional Programming vs. Machine Learning

The clearest way to understand ML is to contrast it with how software has traditionally been written.

**Traditional programming** — you write explicit rules:
```
Data + Rules → Output
```

**Machine learning** — you let the machine discover the rules from examples:
```
Data + Output (labels) → Rules (model)
```

| | Traditional Programming | Machine Learning |
|---|---|---|
| **How rules are created** | Written by humans | Learned from data |
| **Handles complexity** | Breaks down with too many edge cases | Scales with more data |
| **Adaptability** | Must be manually updated | Retrains on new data |
| **Explainability** | Easy — logic is explicit | Varies (linear models = easy, neural nets = hard) |
| **Best for** | Well-defined, stable problems | Pattern-heavy, high-dimensional problems |

### A concrete example

Suppose you want to detect spam email.

The traditional approach: write rules like *"if subject contains 'free money', mark as spam"*. This breaks immediately — spammers change wording constantly, and legitimate emails might trip the filter.

The ML approach: show the model thousands of spam and non-spam emails. It discovers its own patterns — word frequencies, sender behavior, formatting quirks — and generalizes to emails it's never seen before.

## The Three Types of Machine Learning

Almost every ML problem falls into one of three categories, defined by what kind of feedback the model receives during training.

![Types of Machine Learning](https://raw.githubusercontent.com/schemabotview/machine-learning/main/img/01-types-of-ml.svg)

---

### 1. Supervised Learning

The model learns from **labeled examples** — every training sample has an input and a known correct output.

- **Input:** features (e.g., house size, location, age)
- **Output:** label (e.g., house price, or "spam" / "not spam")
- **Goal:** learn a mapping `f(X) → y` that generalizes to unseen inputs

Two sub-types:
- **Regression** — output is a continuous number (e.g., price, temperature, score)
- **Classification** — output is a category (e.g., spam/not-spam, cat/dog, fraud/legit)

**Examples:** email spam detection, house price prediction, medical diagnosis, credit scoring

---

### 2. Unsupervised Learning

The model learns from **unlabeled data** — there are no correct answers to learn from. The goal is to find hidden structure in the data.

- **Input:** features only, no labels
- **Goal:** discover groups, patterns, or compressed representations

Common tasks:
- **Clustering** — group similar items together (e.g., customer segmentation)
- **Dimensionality reduction** — compress data while preserving structure (e.g., PCA for visualization)
- **Anomaly detection** — find unusual points that don't fit the pattern

**Examples:** customer segmentation, topic modeling, fraud detection, recommendation systems

---

### 3. Reinforcement Learning

An **agent** learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a policy that maximizes cumulative reward over time.

- **No labeled dataset** — the model learns from trial and error
- **Feedback:** reward signal (positive or negative) after each action
- **Goal:** learn a strategy (policy) that maximizes long-term reward

**Examples:** game-playing AI (AlphaGo, chess engines), robot locomotion, trading algorithms, LLM fine-tuning with RLHF

---

| | Supervised | Unsupervised | Reinforcement |
|---|---|---|---|
| **Labels needed?** | Yes | No | No (reward signal instead) |
| **Feedback type** | Correct answer | None | Reward / penalty |
| **Common tasks** | Classification, Regression | Clustering, Reduction | Games, Robotics, RLHF |
| **Typical data size** | Medium–Large | Any | Simulated environment |

## When Should You Use Machine Learning?

ML is powerful, but it's not always the right tool. Use it when:

| Condition | Why ML helps |
|---|---|
| The rules are too complex to write manually | ML discovers them from data |
| The problem has too many variables or edge cases | ML handles high-dimensional input naturally |
| You have lots of historical examples | ML can learn from past patterns |
| The environment changes over time | A retrained model adapts; hardcoded rules don't |

**Don't use ML when:**
- A simple rule-based or deterministic algorithm solves the problem cleanly
- You have very little data (< a few hundred examples for most models)
- The decision needs to be 100% explainable and auditable (e.g., legal requirements)
- A lookup table, SQL query, or formula is sufficient

> **Rule of thumb:** reach for ML when *data* is your source of truth, not human expertise.

## Hands-On: Supervised vs. Unsupervised with Fintech Bank Data

Let's see both types of learning in action using data from **Fintech Bank** — the domain we'll use throughout this entire course.

The bank has loan customers. We'll use two tiny datasets:
- **Supervised:** predict a customer's monthly EMI from their loan principal and tenure
- **Unsupervised:** segment customers by credit score and monthly spend — no labels used

In [ ]:
import pandas as pd
import numpy as np

# Fintech Bank — loan_accounts (small in-memory sample)
loans = pd.DataFrame({
    "customer_id":   ["CUST001", "CUST002", "CUST003", "CUST004", "CUST005",
                      "CUST006", "CUST007", "CUST008", "CUST009", "CUST010"],
    "full_name":     ["Aarav Sharma", "Priya Nair", "Rohan Gupta", "Anjali Mehta", "Vikram Reddy",
                      "Sneha Iyer",  "Arjun Verma", "Kavya Pillai", "Dev Joshi",  "Meera Rao"],
    "principal":     [500000, 300000, 750000, 200000, 1000000,
                      450000, 650000, 350000, 800000, 250000],  # ₹ loan amount
    "tenure_months": [60, 36, 84, 24, 120,
                      48, 72, 36, 96, 24],
    "monthly_emi":   [10607, 9374, 10966, 9394, 11000,
                      10255, 10765, 10960, 10667, 11741],       # ₹ — what we want to predict
})

loans[["customer_id", "full_name", "principal", "tenure_months", "monthly_emi"]]

In [ ]:
# --- Supervised Learning: Regression ---
# Goal: learn a rule that predicts monthly_emi from principal + tenure_months
# Model: LinearRegression — the simplest supervised model (a straight-line fit)

from sklearn.linear_model import LinearRegression

X = loans[["principal", "tenure_months"]]   # features — what we know
y = loans["monthly_emi"]                    # label — what we want to predict

model = LinearRegression()
model.fit(X, y)    # learn the mapping: X → y

# Predict EMI for a new customer not in the training data
new_customer = pd.DataFrame({"principal": [600000], "tenure_months": [60]})
predicted_emi = model.predict(new_customer)[0]
print(f"Predicted monthly EMI for ₹6,00,000 over 60 months: ₹{predicted_emi:,.0f}")

In [ ]:
# --- Unsupervised Learning: Clustering ---
# Goal: discover natural customer segments from credit score + monthly spend
# No labels used — the model finds structure on its own

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Fintech Bank — customers with credit_score and avg monthly card spend
customers = pd.DataFrame({
    "customer_id":   loans["customer_id"],
    "credit_score":  [780, 620, 740, 810, 590, 760, 680, 720, 830, 640],
    "monthly_spend": [45000, 12000, 38000, 62000, 8000,
                      51000, 22000, 34000, 70000, 15000],  # ₹ avg card spend
})

# Scale features so credit_score and monthly_spend are on the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers[["credit_score", "monthly_spend"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
customers["segment"] = kmeans.fit_predict(X_scaled)   # no labels passed

print(customers[["customer_id", "credit_score", "monthly_spend", "segment"]])

In [ ]:
import matplotlib.pyplot as plt

segment_labels = {0: "Mid-tier", 1: "Budget", 2: "Premium"}
colors = {0: "#3b82f6", 1: "#f59e0b", 2: "#10b981"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — Supervised: actual vs predicted EMI
axes[0].scatter(loans["principal"], loans["monthly_emi"], color="#3b82f6", label="Actual EMI", zorder=3)
x_line = np.linspace(loans["principal"].min(), loans["principal"].max(), 100).reshape(-1, 1)
# predict across a range of principals at a fixed tenure of 60 months
x_line_full = pd.DataFrame({"principal": x_line.flatten(), "tenure_months": 60})
axes[0].plot(x_line_full["principal"], model.predict(x_line_full), color="#ef4444", label="Linear fit", linewidth=2)
axes[0].set_title("Supervised — Linear Regression\n(predict monthly EMI from principal)")
axes[0].set_xlabel("Loan Principal (₹)")
axes[0].set_ylabel("Monthly EMI (₹)")
axes[0].legend()

# Right — Unsupervised: customer clusters
for seg, label in segment_labels.items():
    mask = customers["segment"] == seg
    axes[1].scatter(
        customers.loc[mask, "credit_score"],
        customers.loc[mask, "monthly_spend"],
        color=colors[seg], label=label, s=80, zorder=3
    )
axes[1].set_title("Unsupervised — KMeans Clustering\n(segment customers, no labels used)")
axes[1].set_xlabel("Credit Score")
axes[1].set_ylabel("Monthly Spend (₹)")
axes[1].legend()

plt.tight_layout()
plt.show()

The left chart shows supervised learning — the model found a line that fits the relationship between loan principal and monthly EMI. Give it a new principal amount and it predicts the EMI.

The right chart shows unsupervised learning — without any labels, KMeans found three natural customer segments: high credit score + high spend (Premium), mid-range (Mid-tier), and low score + low spend (Budget). The bank can now target each group differently.

## Summary

- **Machine learning** lets systems learn rules from data instead of having rules written by hand.
- The key shift: Traditional programming is `Data + Rules → Output`. ML is `Data + Output → Rules`.
- There are three main types: **Supervised** (learns from labels), **Unsupervised** (finds hidden structure), **Reinforcement** (learns from rewards).
- Use ML when the problem is too complex for rules, you have historical data, and the environment changes over time.
- Core vocabulary: features, labels, training/test sets, parameters, hyperparameters, overfitting, generalization.

Next up: **The ML Workflow** — how a real ML project moves from raw data to a deployed model.